# Virtual Try-On API Server
This notebook runs the CatVTON model and exposes an API for your local web app.

**Setup:** Runtime > Change runtime type > **T4 GPU** > Run all cells

## Step 1: Verify GPU

In [ ]:
!nvidia-smi -L
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Go to Runtime > Change runtime type > T4 GPU")
print(f"GPU ready: {torch.cuda.get_device_name(0)}")

## Step 2: Clone Repo and Install Dependencies

In [ ]:
!git clone https://github.com/Zheng-Chong/CatVTON.git
%cd CatVTON
!pip install -q accelerate diffusers transformers peft "huggingface_hub>=0.34.0,<2.0" \
    fvcore cloudpickle omegaconf pycocotools av scikit-image \
    fastapi uvicorn python-multipart nest_asyncio

## Step 3: Load Model

In [ ]:
import os, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
sys.path.insert(0, ".")

import torch
import numpy as np
from PIL import Image
from diffusers.image_processor import VaeImageProcessor
from huggingface_hub import snapshot_download

from model.cloth_masker import AutoMasker, vis_mask
from model.pipeline import CatVTONPipeline
from utils import init_weight_dtype, resize_and_crop, resize_and_padding

# Use the exact training resolution for best color accuracy
WIDTH = 768
HEIGHT = 1024
DEVICE = "cuda"

# Enable memory-efficient attention (Flash Attention on T4)
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)

repo_path = snapshot_download(repo_id="zhengchong/CatVTON")
print(f"Model downloaded to: {repo_path}")

pipeline = CatVTONPipeline(
    base_ckpt="booksforcharlie/stable-diffusion-inpainting",
    attn_ckpt=repo_path,
    attn_ckpt_version="mix",
    weight_dtype=init_weight_dtype("fp16"),
    use_tf32=True,
    device=DEVICE,
    skip_safety_check=True,
)

# VAE memory optimization
pipeline.vae.enable_slicing()

mask_processor = VaeImageProcessor(
    vae_scale_factor=8, do_normalize=False,
    do_binarize=True, do_convert_grayscale=True
)

automasker = AutoMasker(
    densepose_ckpt=os.path.join(repo_path, "DensePose"),
    schp_ckpt=os.path.join(repo_path, "SCHP"),
    device=DEVICE
)

print(f"All models loaded! Resolution: {WIDTH}x{HEIGHT}")
print(f"VRAM used: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

# Warmup run — first inference compiles CUDA kernels and is slow.
# This dummy run pays that cost now so real requests are fast.
import time as _time
print("Running warmup inference (one-time)...")
_t = _time.time()
_dummy_person = Image.new("RGB", (WIDTH, HEIGHT), (128, 128, 128))
_dummy_cloth = Image.new("RGB", (WIDTH, HEIGHT), (200, 200, 200))
_dummy_mask = Image.new("L", (WIDTH, HEIGHT), 255)
_dummy_mask = mask_processor.blur(_dummy_mask, blur_factor=9)
with torch.inference_mode():
    pipeline(
        image=_dummy_person,
        condition_image=_dummy_cloth,
        mask=_dummy_mask,
        num_inference_steps=5,
        guidance_scale=2.5,
        generator=torch.Generator(device=DEVICE).manual_seed(0),
    )
import gc; gc.collect(); torch.cuda.empty_cache()
print(f"Warmup done in {_time.time()-_t:.1f}s — subsequent runs will be faster!")

## Step 4: Define Inference Function

In [ ]:
import io, base64, gc, time as _time

@torch.inference_mode()
def run_try_on(person_image_bytes, cloth_image_bytes, cloth_type="upper",
               num_inference_steps=30, guidance_scale=2.5, seed=42):
    """Optimized inference — uses torch.inference_mode and fewer steps."""
    t0 = _time.time()

    person_image = Image.open(io.BytesIO(person_image_bytes)).convert("RGB")
    cloth_image = Image.open(io.BytesIO(cloth_image_bytes)).convert("RGB")

    person_image = resize_and_crop(person_image, (WIDTH, HEIGHT))
    cloth_image = resize_and_padding(cloth_image, (WIDTH, HEIGHT))
    t1 = _time.time()

    mask = automasker(person_image, cloth_type)["mask"]
    mask = mask_processor.blur(mask, blur_factor=9)
    t2 = _time.time()

    generator = None
    if seed != -1:
        generator = torch.Generator(device=DEVICE).manual_seed(seed)

    result_image = pipeline(
        image=person_image,
        condition_image=cloth_image,
        mask=mask,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=generator,
    )[0]
    t3 = _time.time()

    buffer = io.BytesIO()
    result_image.save(buffer, format="PNG")
    result_b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

    gc.collect()
    torch.cuda.empty_cache()

    print(f"  Resize: {t1-t0:.1f}s | Mask: {t2-t1:.1f}s | Diffusion: {t3-t2:.1f}s | Total: {t3-t0:.1f}s")
    return result_b64

print("Inference function ready (optimized: 30 steps, inference_mode)")

## Step 5: Create API Server

In [ ]:
import time
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import JSONResponse

app = FastAPI(title="CatVTON API")

@app.get("/api/health")
async def health_check():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU"
    return {
        "status": "ok",
        "gpu": gpu_name,
        "resolution": f"{WIDTH}x{HEIGHT}",
    }

@app.post("/api/try-on")
async def try_on(
    person_image: UploadFile = File(...),
    cloth_image: UploadFile = File(...),
    cloth_type: str = Form("upper"),
    num_inference_steps: int = Form(30),
    guidance_scale: float = Form(2.5),
    seed: int = Form(42),
):
    if cloth_type not in ("upper", "lower", "overall"):
        raise HTTPException(status_code=400, detail="cloth_type must be upper, lower, or overall")

    num_inference_steps = max(10, min(100, num_inference_steps))
    guidance_scale = max(0.0, min(7.5, guidance_scale))

    person_bytes = await person_image.read()
    cloth_bytes = await cloth_image.read()

    start_time = time.time()
    try:
        result_b64 = run_try_on(
            person_bytes, cloth_bytes, cloth_type,
            num_inference_steps, guidance_scale, seed
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Inference error: {str(e)}")

    elapsed = round(time.time() - start_time, 2)
    return JSONResponse({
        "status": "success",
        "result_image": result_b64,
        "elapsed_seconds": elapsed,
    })

print("API server defined!")

## Step 6: Start Server with Cloudflare Tunnel

No signup or token needed. Copy the public URL printed below and paste it into your local app.

In [ ]:
import nest_asyncio
import uvicorn
import threading
import subprocess
import re
import time

nest_asyncio.apply()

# Start FastAPI server in background
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)

# Download and start cloudflared tunnel (no signup needed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Start tunnel and capture the URL
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

# Wait for the URL to appear in cloudflared output
public_url = None
for _ in range(30):
    line = proc.stderr.readline()
    match = re.search(r"(https://[a-zA-Z0-9-]+\.trycloudflare\.com)", line)
    if match:
        public_url = match.group(1)
        break

if public_url:
    print()
    print("=" * 60)
    print("  API SERVER IS RUNNING!")
    print(f"  PUBLIC URL: {public_url}")
    print("  Paste this URL into your local app!")
    print("=" * 60)
    print()
else:
    print("Could not get tunnel URL. Check output above for errors.")

# Keep the cell alive so the tunnel stays open
import signal
signal.pause()